# Project 34 — LangGraph Data Cleaning Approval Flow
## Suggest Transforms → Review → Apply with Approval

**Stack:** LangGraph · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langgraph langchain langchain-ollama pandas

## Step 1 — Setup with Dirty Dataset

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

# Create a messy dataset
dirty_data = pd.DataFrame({
    "name": ["John Smith", "jane doe", "  Bob Johnson ", "ALICE WONG", "Charlie", None],
    "email": ["john@email.com", "not-an-email", "bob@email.com", "alice@email.com", "", "charlie@email.com"],
    "age": [25, 30, -5, 150, 28, 22],
    "salary": [50000, 60000, 55000, None, 45000, 52000],
    "dept": ["Engineering", "engineering", "Eng", "ENGINEERING", "Sales", "Sales"],
})
print("Dirty Dataset:")
print(dirty_data.to_string())
print(f"\nIssues: {dirty_data.isnull().sum().sum()} nulls, inconsistent casing, invalid values")

class CleaningState(TypedDict):
    data_summary: str
    suggested_transforms: str
    approved_transforms: list[str]
    cleaning_log: str
    clean_data_summary: str

## Step 2 — Profile Data and Suggest Transforms

In [ ]:
def profile_data(state: CleaningState) -> CleaningState:
    summary = f"""Dataset Shape: {dirty_data.shape}
Columns: {list(dirty_data.columns)}
Nulls: {dirty_data.isnull().sum().to_dict()}
Sample:\n{dirty_data.head().to_string()}"""
    print(f"  📊 Data profiled: {dirty_data.shape}")
    return {"data_summary": summary}

def suggest_transforms(state: CleaningState) -> CleaningState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a data quality expert. Analyze the dataset and suggest
specific cleaning transforms. For each issue, suggest a numbered transform like:
1. [column] — issue — transform
Be specific about what values to change."""),
        ("human", "Dataset summary:\n{summary}")
    ])
    chain = prompt | llm | StrOutputParser()
    suggestions = chain.invoke({"summary": state["data_summary"]})
    print(f"  🔧 Transforms suggested")
    return {"suggested_transforms": suggestions}

def auto_approve(state: CleaningState) -> CleaningState:
    """Simulate approval — in production, this would be a human review step."""
    transforms = [
        "Trim whitespace from name column",
        "Title-case all names",
        "Standardize dept to 'Engineering' or 'Sales'",
        "Replace invalid ages (<0 or >120) with median",
        "Fill null salary with column median",
        "Validate email format",
    ]
    print(f"  ✅ Approved {len(transforms)} transforms")
    return {"approved_transforms": transforms}

def apply_transforms(state: CleaningState) -> CleaningState:
    global dirty_data
    clean = dirty_data.copy()
    log = []

    # Apply each transform
    clean["name"] = clean["name"].fillna("Unknown").str.strip().str.title()
    log.append(f"Names: trimmed, title-cased, filled {dirty_data['name'].isnull().sum()} nulls")

    dept_map = {"eng": "Engineering", "engineering": "Engineering"}
    clean["dept"] = clean["dept"].str.lower().map(lambda x: dept_map.get(x, x.title() if isinstance(x,str) else x))
    log.append("Dept: standardized to consistent casing")

    median_age = clean["age"][clean["age"].between(0, 120)].median()
    invalid_ages = ~clean["age"].between(0, 120)
    clean.loc[invalid_ages, "age"] = int(median_age)
    log.append(f"Age: replaced {invalid_ages.sum()} invalid values with median ({median_age:.0f})")

    median_salary = clean["salary"].median()
    null_salaries = clean["salary"].isnull().sum()
    clean["salary"] = clean["salary"].fillna(median_salary)
    log.append(f"Salary: filled {null_salaries} nulls with median ({median_salary:.0f})")

    import re
    email_pattern = re.compile(r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$')
    invalid_emails = clean["email"].apply(lambda x: not bool(email_pattern.match(str(x))) if x else True)
    clean.loc[invalid_emails, "email"] = "invalid@placeholder.com"
    log.append(f"Email: flagged {invalid_emails.sum()} invalid entries")

    cleaning_log = "\n".join([f"  • {l}" for l in log])
    print(f"  🧹 Applied {len(log)} transforms")
    return {
        "cleaning_log": cleaning_log,
        "clean_data_summary": f"Clean dataset:\n{clean.to_string()}"
    }

## Step 3 — Build Cleaning Graph

In [ ]:
graph = StateGraph(CleaningState)
graph.add_node("profile", profile_data)
graph.add_node("suggest", suggest_transforms)
graph.add_node("approve", auto_approve)
graph.add_node("apply", apply_transforms)

graph.set_entry_point("profile")
graph.add_edge("profile", "suggest")
graph.add_edge("suggest", "approve")
graph.add_edge("approve", "apply")
graph.add_edge("apply", END)

app = graph.compile()
print("Data cleaning workflow compiled!")

## Step 4 — Run the Pipeline

In [ ]:
result = app.invoke({
    "data_summary": "", "suggested_transforms": "",
    "approved_transforms": [], "cleaning_log": "", "clean_data_summary": "",
})

print("\nSuggested Transforms:")
print(result["suggested_transforms"][:500])
print("\nCleaning Log:")
print(result["cleaning_log"])
print("\nResult:")
print(result["clean_data_summary"])

## What You Learned
- **Automated data profiling** to detect issues
- **LLM-suggested transforms** for data quality
- **Approval workflow** before applying changes
- **Structured cleaning log** for auditability